In [1]:
from collections import defaultdict
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


class BooleanRetrieval:
    def __init__(self, documents):
        """Initializes the inverted index for Boolean Retrieval."""
        self.documents = documents
        self.inverted_index = defaultdict(set) # Dictionary to store the inverted index
        self.build_index()

    def build_index(self):
        """Builds the inverted index from the provided documents."""
        for doc_id, text in self.documents.items():
            words = set(text.lower().split())
            for word in words:
                self.inverted_index[word].add(doc_id)

    def boolean_search(self, query):
        """Performs a Boolean search with AND, OR, and NOT operations."""
        terms = query.lower().split()
        result_set = set(self.documents.keys())
        
        operation = "AND" # Default operation
        for term in terms:
            if term == "and":
                operation = "AND"
            elif term == "or":
                operation = "OR"
            elif term == "not":
                operation = "NOT"
            else:
                term_docs = self.inverted_index.get(term, set())
                
                if operation == "AND":
                    result_set &= term_docs
                elif operation == "OR":
                    result_set |= term_docs
                elif operation == "NOT":
                    result_set -= term_docs
        return result_set


class VectorSpaceRetrieval:
    def __init__(self, documents):
        """Initializes the TF-IDF vectorization for document retrieval."""
        self.documents = documents
        self.vectorizer = TfidfVectorizer()
        self.doc_ids = list(documents.keys())
        self.doc_vectors = self.vectorizer.fit_transform(documents.values())

    def vector_search(self, query):
        """Performs a vector space search using cosine similarity."""
        query_vector = self.vectorizer.transform([query])
        similarities = np.dot(self.doc_vectors, query_vector.T).toarray().flatten()
        ranked_results = sorted(zip(self.doc_ids, similarities), key=lambda x: x[1], reverse=True)
        return ranked_results


documents = {
    1: "Web content extraction involves retrieving structured data",
    2: "Search engines use document indexing for efficient retrieval",
    3: "Document retrieval is important in web mining applications",
    4: "Indexing helps in retrieving relevant documents based on query terms"
}

boolean_index = BooleanRetrieval(documents)
boolean_queries = ["retrieval AND document", "document OR indexing", "retrieval NOT indexing"]

print("\n=== Boolean Retrieval Results ===")
for query in boolean_queries:
    result = boolean_index.boolean_search(query)
    print(f"Query: '{query}' -> Documents: {sorted(result) if result else 'No matching documents'}")

vector_index = VectorSpaceRetrieval(documents)
vector_queries = ["document retrieval", "web mining", "structured data"]

print("\n=== Vector Space Model Results ===")
for query in vector_queries:
    result = vector_index.vector_search(query)
    print(f"Query: '{query}' -> Ranked Documents: {[(doc, round(score, 4)) for doc, score in result if score > 0]}")


=== Boolean Retrieval Results ===
Query: 'retrieval AND document' -> Documents: [2, 3]
Query: 'document OR indexing' -> Documents: [2, 3, 4]
Query: 'retrieval NOT indexing' -> Documents: [3]

=== Vector Space Model Results ===
Query: 'document retrieval' -> Ranked Documents: [(3, np.float64(0.4378)), (2, np.float64(0.4256))]
Query: 'web mining' -> Ranked Documents: [(3, np.float64(0.5)), (1, np.float64(0.1954))]
Query: 'structured data' -> Ranked Documents: [(1, np.float64(0.566))]
